<a href="https://colab.research.google.com/github/aniketh703/legal-contract-analyzer/blob/main/kind-newton-jsmo1r/notebooks/finetune_inlegalbert_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tune InLegalBERT — Legal Contract Analyzer

Runs `pipeline/train_classifier_bert.py` from the repo on a free Colab GPU instead of the 12+ hour CPU estimate in the script's own docstring.

**Before running anything below:** `Runtime -> Change runtime type -> T4 GPU`, then `Save`.

What this notebook does:
1. Clones the repo's `claude/kind-newton-jsmo1r` branch (has the fine-tuning script + the fixed 47-class held-out evaluation).
2. Pulls `legal_contract_clauses.csv` (the CUAD training data) from the `main` branch of the same repo, since `pipeline/train_classifier.py` expects it one directory above the repo root.
3. Installs the handful of extra packages needed (Colab already ships torch + transformers).
4. Runs the fine-tune, which fine-tunes `law-ai/InLegalBERT` on the *exact* held-out split the TF-IDF+LR baseline was scored on, and writes `evaluation/results/inlegalbert_47class.json`.
5. Prints a baseline-vs-InLegalBERT comparison and downloads the results JSON.

Expect roughly 15-40 minutes on a T4 for 4 epochs over ~7,758 training rows (vs. 12+ hours on CPU).

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'No GPU detected -- go to Runtime > Change runtime type > T4 GPU, then re-run this cell.'

Tesla T4, 15360 MiB


In [2]:
%cd /content
!rm -rf legal-contract-analyzer
!git clone -b claude/kind-newton-jsmo1r https://github.com/aniketh703/legal-contract-analyzer.git
%cd legal-contract-analyzer

/content
Cloning into 'legal-contract-analyzer'...
remote: Enumerating objects: 177, done.
remote: Counting objects: 100% (177/177), done.
remote: Compressing objects: 100% (119/119), done.
remote: Total 177 (delta 52), reused 167 (delta 42), pack-reused 0 (from 0)
Receiving objects: 100% (177/177), 14.45 MiB | 15.67 MiB/s, done.
Resolving deltas: 100% (52/52), done.
/content/legal-contract-analyzer


In [3]:
# The CUAD training CSV lives on the `main` branch (report-assets branch), not on
# the code branch. pipeline/train_classifier.py expects it one directory above
# the repo root (CSV_PATH = ROOT.parent / "legal_contract_clauses.csv"), i.e.
# /content/legal_contract_clauses.csv when this repo is cloned to /content/legal-contract-analyzer.
!git fetch origin main --depth 1
!git show origin/main:legal_contract_clauses.csv > ../legal_contract_clauses.csv
!wc -l ../legal_contract_clauses.csv

remote: Total 0 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
From https://github.com/aniketh703/legal-contract-analyzer
 * branch            main       -> FETCH_HEAD
11926 ../legal_contract_clauses.csv


In [4]:
# Minimal install -- skip requirements.txt's heavier/unrelated deps
# (label-studio, faiss-cpu, pdfplumber, pymupdf, pytesseract) that this
# training script doesn't touch and that would just slow the install down.
!pip install -q -U transformers accelerate scikit-learn joblib pandas numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 114.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 128.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 108.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 107.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.5 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.
numba 0.61.2 requires numpy<2.3,>=1.24, but you have numpy 

In [5]:
!python pipeline/train_classifier_bert.py

Traceback (most recent call last):
  File "/content/legal-contract-analyzer/pipeline/train_classifier_bert.py", line 62, in <module>
    from pipeline.train_classifier import normalize_type, CSV_PATH, INDIAN_CSV_PATH
  File "/content/legal-contract-analyzer/pipeline/pipeline.py", line 15, in <module>
    from pipeline.segmenter import segment_contract, load_contract_text
ModuleNotFoundError: No module named 'pipeline.segmenter'; 'pipeline' is not a package


In [6]:
import json

with open("evaluation/results/tfidf_lr_baseline_47class.json") as f:
    baseline = json.load(f)
with open("evaluation/results/inlegalbert_47class.json") as f:
    bert = json.load(f)

b_cr = baseline["classification_report"]
n_cr = bert["classification_report"]

print(f"{'Metric':<22}{'TF-IDF + LR':>14}{'InLegalBERT':>14}")
print(f"{'Accuracy':<22}{baseline['accuracy']:>14.4f}{bert['accuracy']:>14.4f}")
print(f"{'Macro F1':<22}{b_cr['macro avg']['f1-score']:>14.4f}{n_cr['macro avg']['f1-score']:>14.4f}")
print(f"{'Weighted F1':<22}{b_cr['weighted avg']['f1-score']:>14.4f}{n_cr['weighted avg']['f1-score']:>14.4f}")

FileNotFoundError: [Errno 2] No such file or directory: 'evaluation/results/inlegalbert_47class.json'

In [ ]:
from google.colab import files

files.download("evaluation/results/inlegalbert_47class.json")

## Optional: keep the fine-tuned model weights

Only run this if you want to reuse the fine-tuned model later (e.g. to wire it into the live demo) -- it's a few hundred MB, so skip it if you only need the numbers above.

In [ ]:
!zip -qr inlegalbert_classifier.zip models/inlegalbert_classifier
from google.colab import files

files.download("inlegalbert_classifier.zip")